# Exploratory Data Analysis with SQL

This notebook performs lightweight exploratory analysis using **SQL** on the labeled launch dataset created in **Step 03**.

**Why SQL here?**

- To demonstrate how the dataset can be explored using SQL-style aggregations and filters
- To mirror how analysts often validate patterns in a database (counts, group-bys, date filters)

## Context
SpaceX's ability to reuse the first stage is a major driver of launch cost reduction.
If we can understand which factors are associated with successful landings, we can better frame the prediction problem tackled later in the project.

## Data used

**Input:** `../data/processed/03_dataset_part_2.csv` cleaned launch-level dataset with a binary landing-success lable (`Class`)

**Outputs:** `../data/processed/04_spacex_eda.sqlite` SQLite database created from the CSV for ad-hoc querying

## Notebook outline

1. Setup
2. Load labeled dataset from Step 03
3. Create a local SQLite database + load the table
4. SQL EDA queries

## 1. Setup
We load the processed CSV from step 03, create a local SQLite database in `../data/processed/`, and expose a small helper for running SQL queries.

In [2]:
from pathlib import Path
import sqlite3
import pandas as pd

# Paths
PROCESSED_DIR = Path('../data/processed')
INPUT_CSV = PROCESSED_DIR / '03_dataset_part_2.csv'
DB_PATH = PROCESSED_DIR / '04_spacex_launches.db'

# Ensure output directory exists
PROCESSED_DIR.mkdir(parents = True, exist_ok = True)

# SQLite connection
con = sqlite3.connect(DB_PATH)
con.row_factory = sqlite3.Row

def q(sql: str, params = None) -> pd.DataFrame:
    """Run an SQL query and return the result as a pandas DataFrame."""
    return pd.read_sql_query(sql, con, params = params)

## 1. Load labeled dataset

We load the cleaned + labeled dataset from Step 03 that includes the binary target `Class` (1 = successful landing, 0 = unsuccessful/unknown).

In [24]:
df = pd.read_csv(INPUT_CSV)

# Standardize column names for SQL
rename_map = {
    'FlightNumber': 'flight_number',
    'Date': 'date',
    'BoosterVersion': 'booster_version',
    'PayloadMass': 'payload_mass_kg',
    'Orbit': 'orbit',
    'LaunchSite': 'launch_site',
    'Outcome': 'landing_outcome',
    'Flights': 'flights',
    'GridFins': 'grid_fins',
    'Reused': 'reused',
    'Legs': 'legs',
    'LandingPad': 'landing_pad',
    'Block': 'block',
    'ReusedCount': 'reused_count',
    'Serial': 'serial',
    'Longitude': 'longitude',
    'Latitude': 'latitude',
    'Class': 'class',
}

df_sql = df.rename(columns = rename_map).copy()

# Write to a staging table
df_sql.to_sql('spacex_tbl', con, if_exists = 'replace', index = False, method = 'multi')

90

## 2. Build the analysis table

The course notebook creates a second table that excludes blank `Date` rows.
We keep the same idea here: `spacex_table` is the clean table used by all queries below.

In [25]:
# Drop analysis table if it already exists
con.execute('DROP TABLE IF EXISTS spacex_table;')
con.commit()

In [26]:
# Create the analysis table (exclude records without a date)
con.execute("""
CREATE TABLE spacex_table AS
SELECT *
FROM spacex_tbl
WHERE date IS NOT NULL;
""")
con.commit()

# Helpful indexes for repeated filtering
con.execute('CREATE INDEX IF NOT EXISTS idx_spacex_date ON spacex_table(date);')
con.execute('CREATE INDEX IF NOT EXISTS idx_spacex_launch_site ON spacex_table(launch_site);')
con.execute('CREATE INDEX IF NOT EXISTS idx_spacex_orbit ON spacex_table(orbit);')
con.execute('CREATE INDEX IF NOT EXISTS idx_spacex_class ON spacex_table(class);')
con.commit()

q('SELECT COUNT(*) AS n_rows FROM spacex_table;')

,n_rows
0,90


## 3. Exploratory SQL Questions
Below are targeted SQL queries to understand how landing success varies by **launch site**, **orbit**, **payload mass**, and **time**.

#### 3.1 Unique launch sites in the dataset

In [27]:
q("""
  SELECT DISTINCT launch_site
  FROM spacex_table
  ORDER BY launch_site;
""")

,launch_site
0,CCSFS SLC 40
1,KSC LC 39A
2,VAFB SLC 4E


#### 3.2 Launche sites starting with `CC`

In [28]:
q("""
  SELECT *
  FROM spacex_table
  WHERE launch_site LIKE 'CC%'
  LIMIT 5;
""")

,flight_number,date,booster_version,payload_mass_kg,orbit,launch_site,landing_outcome,flights,grid_fins,reused,legs,landing_pad,block,reused_count,serial,longitude,latitude,class
0,1,2010-06-04,Falcon 9,6123.547647,LEO,CCSFS SLC 40,None None,1,0,0,0,None,1.0,0,B0003,-80.577366,28.561857,0
1,2,2012-05-22,Falcon 9,525.000000,LEO,CCSFS SLC 40,None None,1,0,0,0,None,1.0,0,B0005,-80.577366,28.561857,0
2,3,2013-03-01,Falcon 9,677.000000,ISS,CCSFS SLC 40,None None,1,0,0,0,None,1.0,0,B0007,-80.577366,28.561857,0
3,5,2013-12-03,Falcon 9,3170.000000,GTO,CCSFS SLC 40,None None,1,0,0,0,None,1.0,0,B1004,-80.577366,28.561857,0
4,6,2014-01-06,Falcon 9,3325.000000,GTO,CCSFS SLC 40,None None,1,0,0,0,None,1.0,0,B1005,-80.577366,28.561857,0


#### 3.3 Total payload mass by launch site
This helps compare how launch sites differ in the amount of payload they tend to support.

In [29]:
q("""
  SELECT
    launch_site,
    ROUND(SUM(payload_mass_kg), 1) AS total_payload_kg,
    COUNT(*) AS n_launches
  FROM spacex_table
  GROUP BY launch_site
  ORDER BY total_payload_kg DESC;
""")

,launch_site,total_payload_kg,n_launches
0,CCSFS SLC 40,305987.2,55
1,KSC LC 39A,168179.1,22
2,VAFB SLC 4E,76953.0,13


#### 3.4 Average payload mass by orbit
Orbit is one of the strongest candidates for explaining mission configuration differences.

In [30]:
q("""
  SELECT
    orbit,
    ROUND(AVG(payload_mass_kg), 1) AS avg_payload_kg,
    COUNT(*) AS n_launches
  FROM spacex_table
  GROUP BY orbit
  ORDER BY avg_payload_kg DESC;
""")

,orbit,avg_payload_kg,n_launches
0,VLEO,15428.6,14
1,PO,7583.7,9
2,GEO,6123.5,1
3,SO,6123.5,1
4,GTO,5012.0,27
5,MEO,3987.0,3
6,LEO,3890.8,7
7,ISS,3279.9,21
8,SSO,2060.0,5
9,ES-L1,570.0,1


#### 3.5 First observed successful landing date

We treat `class = 1` as a successful first-stage landing.

In [31]:
q("""
  SELECT MIN(date) AS first_success_date
  FROM spacex_table
  WHERE class = 1;
""")

,first_success_date
0,2014-04-18


#### 3.6 Successful drone-ship landings within a payload band
Drone-ship landings correspond to `'ASDS'` in `landing_outcome`.

We filter to successful cases and a mid-range payload band (4,000-6,000 kg).

In [32]:
q("""
  SELECT DISTINCT booster_version
  FROM spacex_table
  WHERE landing_outcome LIKE '%ASDS%'
    AND class = 1
    AND payload_mass_kg BETWEEN 4000 and 6000
  ORDER BY booster_version;
""")

,booster_version
0,Falcon 9


#### 3.7 Success vs. failure counts
A quick view of class balance in the dataset.

In [33]:
q("""
  SELECT
    CASE WHEN class = 1 THEN 'success' ELSE 'failure' END AS outcome,
    COUNT(*) AS n
  FROM spacex_table
  GROUP BY outcome
  ORDER BY n DESC;
""")

,outcome,n
0,success,60
1,failure,30


#### 3.8 Booster version(s) that carried the maximum payload
Uses a subquery to match the maximum observed payload mass.

In [34]:
q("""
  SELECT DISTINCT booster_version, payload_mass_kg
  FROM spacex_table
  WHERE payload_mass_kg = (SELECT MAX(payload_mass_kg) FROM spacex_table);
""")

,booster_version,payload_mass_kg
0,Falcon 9,15600.0


#### 3.9 Failed drone-ship landings in 2015 (month level)
SQLite doesn't have a built-in 'month name' function, so we extract the month using `substr(date, 6, 2)`.

In [35]:
q("""
  SELECT
    substr(date, 6, 2) AS month,
    date,
    launch_site,
    orbit,
    landing_outcome,
    payload_mass_kg
  FROM spacex_table
  WHERE substr(date, 1, 4) = '2015'
    AND landing_outcome LIKE '%ASDS%'
    AND class = 0
  ORDER BY date;
""")

,month,date,launch_site,orbit,landing_outcome,payload_mass_kg
0,01,2015-01-10,CCSFS SLC 40,ISS,False ASDS,2395.0
1,04,2015-04-14,CCSFS SLC 40,ISS,False ASDS,1898.0
2,06,2015-06-28,CCSFS SLC 40,ISS,None ASDS,2477.0


#### 3.10 Landing outcome frequency within a historical window

Counts each `landing_outcome` between 2010-06-04 and 2017-03-20 and ranks them in descending order.

In [36]:
q("""
  SELECT landing_outcome, COUNT(*) AS count
  FROM spacex_table
  WHERE date BETWEEN '2010-06-04' AND '2017-03-20'
  GROUP BY landing_outcome
  ORDER BY count DESC;
""")

,landing_outcome,count
0,None None,9
1,True ASDS,5
2,False ASDS,4
3,True RTLS,3
4,True Ocean,3
5,None ASDS,2
6,False Ocean,2


## Quick takeaways (SQL EDA)

A few patterns worth carrying into the visualization + modeling steps:

- Launces are concentrated across a small set of launch sites.
- Orbit and payload mass vary meaningfully across missions, and are likely informative features.
- Landing outcomes include multiple recovery modes (e.g. ASDS vs RTLS), which can be analyzed separately.

## 4. Export

In [37]:
launch_site_summary = q("""
                        SELECT
                            launch_site,
                            COUNT(*) AS n_launches,
                            SUM(class) AS n_success,
                            ROUND(1.0 * SUM(class) / COUNT(*), 3) AS success_rate
                        FROM spacex_table
                        GROUP BY launch_site
                        ORDER BY success_rate DESC, n_launches DESC;
                    """)

out_csv = PROCESSED_DIR / '04_launch_site_success_rate.csv'
launch_site_summary.to_csv(out_csv, index = False)

launch_site_summary

,launch_site,n_launches,n_success,success_rate
0,KSC LC 39A,22,17,0.773
1,VAFB SLC 4E,13,10,0.769
2,CCSFS SLC 40,55,33,0.600
